# Notebook 4 — Deep Learning
### BNP Paribas — Senior AI Engineer

---
## Sommaire
1. [Réseaux de neurones : fondements](#nn)
2. [Backpropagation & Chain Rule](#backprop)
3. [Fonctions d’activation](#activations)
4. [Optimiseurs (SGD, Momentum, Adam, AdamW)](#optimizers)
5. [Régularisation (Dropout, BatchNorm, L1/L2, Early Stopping)](#regularization)
6. [CNN — Convolutional Neural Networks](#cnn)
7. [RNN, LSTM, GRU](#rnn)
8. [Questions d’entretien](#questions)


---
## 1. Réseaux de neurones : fondements <a name='nn'></a>

### Un neurone
`z = w^T x + b`  puis  `a = f(z)`

### Forward Pass
```
Input -> [Linear -> Activation] x L -> Output -> Loss
```

### Architecture
| Couche | Paramètres | Output shape |
|---|---|---|
| Dense(n_in, n_out) | W:(n_out, n_in), b:(n_out,) | (batch, n_out) |
| Conv2D(C_in, C_out, k) | W:(C_out, C_in, k, k), b:(C_out,) | (batch, C_out, H', W') |
| BatchNorm(n) | gamma:(n,), beta:(n,) | same |

### Initialisation des poids
| Methode | Formule | Pour |
|---|---|---|
| Xavier/Glorot | N(0, 2/(n_in+n_out)) | tanh, sigmoid |
| He | N(0, 2/n_in) | ReLU, Leaky ReLU |
| Zeros | 0 | Biais uniquement |


In [ ]:
import numpy as np

# ============================================================
# RESEAU DE NEURONES FROM SCRATCH -- MLP complet
# ============================================================

class DenseLayer:
    """Couche fully-connected avec forward et backward"""
    def __init__(self, n_in, n_out, activation="relu"):
        # He initialization pour ReLU
        self.W = np.random.randn(n_out, n_in) * np.sqrt(2.0 / n_in)
        self.b = np.zeros(n_out)
        self.activation = activation
        self.cache = {}   # pour le backward

    def forward(self, x):
        z = x @ self.W.T + self.b
        self.cache["x"] = x
        self.cache["z"] = z
        if self.activation == "relu":
            a = np.maximum(0, z)
        elif self.activation == "sigmoid":
            a = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
        elif self.activation == "tanh":
            a = np.tanh(z)
        elif self.activation == "linear":
            a = z
        else:
            raise ValueError(f"Unknown activation: {self.activation}")
        self.cache["a"] = a
        return a

    def backward(self, da):
        z = self.cache["z"]
        x = self.cache["x"]
        # Gradient de l activation
        if self.activation == "relu":
            dz = da * (z > 0)
        elif self.activation == "sigmoid":
            s = self.cache["a"]
            dz = da * s * (1 - s)
        elif self.activation == "tanh":
            dz = da * (1 - self.cache["a"]**2)
        elif self.activation == "linear":
            dz = da
        # Gradients des parametres
        self.dW = dz.T @ x / len(x)
        self.db = dz.mean(axis=0)
        return dz @ self.W   # gradient vers la couche precedente

class MLP:
    """Multi-Layer Perceptron"""
    def __init__(self, layer_dims, activations):
        assert len(activations) == len(layer_dims) - 1
        self.layers = [
            DenseLayer(layer_dims[i], layer_dims[i+1], activations[i])
            for i in range(len(activations))
        ]

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, dloss):
        for layer in reversed(self.layers):
            dloss = layer.backward(dloss)

    def update(self, lr):
        for layer in self.layers:
            layer.W -= lr * layer.dW
            layer.b -= lr * layer.db

# Donnees XOR (non lineairement separable)
np.random.seed(42)
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([[0],[1],[1],[0]], dtype=float)

# MLP: 2 -> 8 -> 4 -> 1
mlp = MLP([2, 8, 4, 1], ["relu", "relu", "sigmoid"])

def bce_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-10, 1-1e-10)
    return -np.mean(y_true * np.log(y_pred) + (1-y_true) * np.log(1-y_pred))

print("=== MLP from scratch sur XOR ===")
losses = []
for epoch in range(2000):
    out = mlp.forward(X_xor)
    loss = bce_loss(y_xor, out)
    losses.append(loss)
    # Backward: gradient de BCE
    dl = (out - y_xor) / (out * (1-out) + 1e-10) / len(X_xor)
    mlp.backward(dl)
    mlp.update(lr=0.05)

print(f"Loss initiale  : {losses[0]:.4f}")
print(f"Loss finale    : {losses[-1]:.4f}")
predictions = (mlp.forward(X_xor) > 0.5).astype(int)
print(f"Predictions XOR: {predictions.ravel()}")
print(f"Ground truth   : {y_xor.ravel().astype(int)}")
print(f"Accuracy       : {(predictions == y_xor).mean():.2f}")


---
## 2. Backpropagation & Chain Rule <a name='backprop'></a>

### Principe
La **regle de la chaine** permet de calculer le gradient de la loss par rapport a chaque parametre :

```
dL/dW[l] = dL/da[l] * da[l]/dz[l] * dz[l]/dW[l]
```

### Etapes
1. **Forward pass** : calculer et stocker toutes les activations
2. **Backward pass** : propager le gradient depuis la loss vers les entrees

### Formules cles (couche Dense)
```
z = W*x + b
dL/dW = dL/dz * x^T      (outer product)
dL/db = dL/dz
dL/dx = W^T * dL/dz       (propagation vers la couche precedente)
```

### Vanishing / Exploding Gradients
- **Vanishing** : gradients -> 0, couches profondes n'apprennent plus (sigmoid/tanh, reseaux profonds)
  - Solution : ReLU, BatchNorm, skip connections (ResNet), LSTM
- **Exploding** : gradients -> inf, instabilite numerique
  - Solution : Gradient clipping, weight initialization


In [ ]:
import numpy as np

# ============================================================
# BACKPROP -- verification numerique du gradient
# ============================================================
def numerical_gradient(f, params, h=1e-5):
    """Calcule le gradient numerique par differences finies centrales"""
    grads = {}
    for key, val in params.items():
        grad = np.zeros_like(val)
        it = np.nditer(val, flags=["multi_index"])
        while not it.finished:
            idx = it.multi_index
            old = val[idx]
            val[idx] = old + h
            loss_plus = f(params)
            val[idx] = old - h
            loss_minus = f(params)
            val[idx] = old
            grad[idx] = (loss_plus - loss_minus) / (2*h)
            it.iternext()
        grads[key] = grad
    return grads

# Exemple: une couche sigmoid + BCE loss
np.random.seed(42)
X = np.random.randn(4, 3)
y = np.array([1.0, 0.0, 1.0, 1.0])
params = {"W": np.random.randn(3, 3)*0.1, "b": np.zeros(3)}

def forward_loss(params):
    z = X @ params["W"] + params["b"]
    a = 1 / (1 + np.exp(-z))    # sigmoid
    out = a.sum(axis=1) / 3     # aggregation
    out = np.clip(out, 1e-10, 1-1e-10)
    return -np.mean(y * np.log(out) + (1-y) * np.log(1-out))

# Gradient analytique
z = X @ params["W"] + params["b"]
a = 1 / (1 + np.exp(-z))
out = a.sum(axis=1) / 3
dout = -(y/(out+1e-10) - (1-y)/(1-out+1e-10)) / len(y)
da = dout[:, np.newaxis] * np.ones((len(y), 3)) / 3
dz = da * a * (1 - a)
dW_analytical = X.T @ dz
db_analytical = dz.sum(axis=0)

# Gradient numerique
num_grads = numerical_gradient(forward_loss, params)

print("=== Verification du gradient (gradient check) ===")
for key, analytical, numerical in [
    ("dW", dW_analytical, num_grads["W"]),
    ("db", db_analytical, num_grads["b"])
]:
    err = np.linalg.norm(analytical - numerical) / (np.linalg.norm(analytical) + np.linalg.norm(numerical) + 1e-10)
    print(f"  {key}: relative error = {err:.2e}  {'OK' if err < 1e-4 else 'ECHEC'}")

# ============================================================
# VANISHING GRADIENTS -- demonstration
# ============================================================
print("\n=== Vanishing Gradients (sigmoid vs ReLU) ===")
def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-500,500)))
def sigmoid_grad(z): s=sigmoid(z); return s*(1-s)
def relu_grad(z): return (z > 0).astype(float)

x = np.random.randn(1, 10)
L = 20  # 20 couches
grad_sig = np.ones_like(x)
grad_rel = np.ones_like(x)
for l in range(L):
    z = np.random.randn(*x.shape)
    grad_sig *= sigmoid_grad(z)
    grad_rel *= relu_grad(z)

print(f"  Gradient norme apres {L} couches sigmoid : {np.linalg.norm(grad_sig):.2e}")
print(f"  Gradient norme apres {L} couches relu    : {np.linalg.norm(grad_rel):.2e}")
print("  => Sigmoid: vanishing. ReLU: stable (mais peut 'mourir' si toujours negatif)")


---
## 3. Fonctions d’activation <a name='activations'></a>

| Fonction | Formule | Range | Gradient | Probleme |
|---|---|---|---|---|
| Sigmoid | 1/(1+e^-z) | (0,1) | s(1-s) | Vanishing gradient, non centre en 0 |
| Tanh | (e^z - e^-z)/(e^z + e^-z) | (-1,1) | 1-tanh^2 | Vanishing gradient |
| ReLU | max(0,z) | [0,+inf) | 0 ou 1 | Dying ReLU (neurones morts) |
| Leaky ReLU | max(alpha*z, z) | (-inf,+inf) | alpha ou 1 | Fixe le dying ReLU |
| ELU | z si z>0, alpha(e^z-1) si z<=0 | (-alpha,+inf) | Lisse | Plus lent |
| GELU | z*Phi(z) | - | - | Transformers (BERT, GPT) |
| Softmax | e^zi / sum(e^zj) | (0,1), sum=1 | - | Derniere couche classif multi |

### Quand utiliser quelle activation ?
- **Couches cachees** : ReLU par defaut, GELU pour transformers
- **Sortie binaire** : Sigmoid
- **Sortie multi-classe** : Softmax
- **Regression** : Lineaire (pas d activation)


In [ ]:
import numpy as np

# ============================================================
# FONCTIONS D ACTIVATION -- implementation et proprietes
# ============================================================
def sigmoid(z):   return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def tanh(z):      return np.tanh(z)
def relu(z):      return np.maximum(0, z)
def leaky_relu(z, alpha=0.01): return np.where(z > 0, z, alpha * z)
def elu(z, alpha=1.0): return np.where(z > 0, z, alpha * (np.exp(np.clip(z,-100,0)) - 1))
def gelu(z):
    """Gaussian Error Linear Unit -- utilise dans BERT, GPT"""
    return 0.5 * z * (1 + np.tanh(np.sqrt(2/np.pi) * (z + 0.044715 * z**3)))
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))  # stabilite numerique
    return e / e.sum(axis=-1, keepdims=True)

# Gradients
def sigmoid_grad(z):    s = sigmoid(z); return s * (1-s)
def tanh_grad(z):       return 1 - tanh(z)**2
def relu_grad(z):       return (z > 0).astype(float)
def leaky_relu_grad(z, alpha=0.01): return np.where(z > 0, 1.0, alpha)

z = np.array([-3, -1, 0, 1, 3], dtype=float)
print("=== Valeurs des activations ===")
print(f"  z         : {z}")
print(f"  sigmoid   : {sigmoid(z).round(4)}")
print(f"  tanh      : {tanh(z).round(4)}")
print(f"  ReLU      : {relu(z).round(4)}")
print(f"  Leaky ReLU: {leaky_relu(z).round(4)}")
print(f"  ELU       : {elu(z).round(4)}")
print(f"  GELU      : {gelu(z).round(4)}")

print("\n=== Gradients ===")
print(f"  sigmoid'  : {sigmoid_grad(z).round(4)}")
print(f"  tanh'     : {tanh_grad(z).round(4)}")
print(f"  ReLU'     : {relu_grad(z).round(4)}")
print("  => sigmoid/tanh: gradient < 0.25 partout => vanishing")
print("  => ReLU: gradient = 0 si z<0 => dying ReLU")

print("\n=== Softmax (multi-classe) ===")
logits = np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]])
probs = softmax(logits)
print(f"  Logits: {logits[0]}")
print(f"  Probs : {probs[0].round(4)}  (sum={probs[0].sum():.4f})")
print(f"  Probs2: {probs[1].round(4)}  (sum={probs[1].sum():.4f})")

print("\n=== Dying ReLU -- probleme ===")
# Si tous les inputs sont negatifs, le gradient est 0 => neurone mort
z_neg = np.array([-2.0, -1.0, -0.5, -0.1])
print(f"  Entrees negatives: {z_neg}")
print(f"  ReLU output: {relu(z_neg)}  => tout a 0!")
print(f"  ReLU gradient: {relu_grad(z_neg)}  => plus de signal!")
print(f"  Leaky ReLU: {leaky_relu(z_neg).round(4)}  => toujours du signal")


---
## 4. Optimiseurs <a name='optimizers'></a>

### Formules
**SGD** : `w = w - lr * grad`

**Momentum** : `v = beta*v + grad ; w = w - lr*v`

**RMSProp** :
```
v = beta*v + (1-beta)*grad^2
w = w - lr * grad / (sqrt(v) + eps)
```

**Adam** :
```
m = beta1*m + (1-beta1)*grad          # 1er moment (moyenne)
v = beta2*v + (1-beta2)*grad^2        # 2e moment (variance non centree)
m_hat = m / (1 - beta1^t)            # correction du biais
v_hat = v / (1 - beta2^t)
w = w - lr * m_hat / (sqrt(v_hat) + eps)
```

**AdamW** : Adam + weight decay **correct** (applique directement sur w, pas dans le gradient)
`w = w * (1 - lr*lambda) - lr * m_hat / (sqrt(v_hat) + eps)`

### Comparaison
| Optimiseur | beta1 | beta2 | lr par defaut | Notes |
|---|---|---|---|---|
| SGD | - | - | 0.01 | Simple, bon avec Momentum+scheduler |
| Adam | 0.9 | 0.999 | 0.001 | Defaut pour la plupart des taches |
| AdamW | 0.9 | 0.999 | 0.001 | Meilleur pour les LLMs (weight decay propre) |
| RMSProp | - | 0.99 | 0.001 | Historiquement bon pour les RNN |


In [ ]:
import numpy as np

# ============================================================
# OPTIMISEURS -- comparaison sur une fonction test
# ============================================================
def rosenbrock(w):
    """Fonction de Rosenbrock: minimum global en (1,1), f=0"""
    return (1 - w[0])**2 + 100*(w[1] - w[0]**2)**2

def rosenbrock_grad(w):
    dw0 = -2*(1 - w[0]) - 400*w[0]*(w[1] - w[0]**2)
    dw1 = 200*(w[1] - w[0]**2)
    return np.array([dw0, dw1])

class SGDOptimizer:
    def __init__(self, lr=0.001): self.lr = lr
    def step(self, w, grad): return w - self.lr * grad

class MomentumOptimizer:
    def __init__(self, lr=0.001, beta=0.9):
        self.lr, self.beta, self.v = lr, beta, None
    def step(self, w, grad):
        if self.v is None: self.v = np.zeros_like(w)
        self.v = self.beta * self.v + grad
        return w - self.lr * self.v

class AdamOptimizer:
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.m = self.v = None
        self.t = 0
    def step(self, w, grad):
        if self.m is None:
            self.m, self.v = np.zeros_like(w), np.zeros_like(w)
        self.t += 1
        self.m = self.beta1*self.m + (1-self.beta1)*grad
        self.v = self.beta2*self.v + (1-self.beta2)*grad**2
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        return w - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

class AdamWOptimizer(AdamOptimizer):
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.01):
        super().__init__(lr, beta1, beta2, eps)
        self.wd = wd
    def step(self, w, grad):
        w_new = super().step(w, grad)
        return w_new - self.lr * self.wd * w  # weight decay separe

print("=== Comparaison des optimiseurs (Rosenbrock) ===")
print(f"  Minimum global: f(1,1) = 0")
print(f"  {'Optimiseur':20s}  {'Loss finale':>12}  {'Dist (1,1)':>12}  {'Iters':>6}")

optimizers = [
    ("SGD (lr=1e-4)",       SGDOptimizer(lr=1e-4)),
    ("SGD (lr=1e-3)",       SGDOptimizer(lr=1e-3)),
    ("Momentum (lr=1e-3)",  MomentumOptimizer(lr=1e-3)),
    ("Adam (lr=1e-2)",      AdamOptimizer(lr=1e-2)),
    ("AdamW (lr=1e-2)",     AdamWOptimizer(lr=1e-2, wd=0.001)),
]

for name, opt in optimizers:
    w = np.array([-1.5, 0.5])  # meme point de depart
    for i in range(5000):
        g = rosenbrock_grad(w)
        w = opt.step(w, g)
    loss = rosenbrock(w)
    dist = np.linalg.norm(w - np.array([1.0, 1.0]))
    print(f"  {name:20s}  {loss:12.4f}  {dist:12.6f}  {5000:>6}")


In [ ]:
import numpy as np

# ============================================================
# LEARNING RATE SCHEDULERS
# ============================================================
def lr_step(epoch, base_lr=0.1, step_size=30, gamma=0.1):
    """Reduit lr par gamma tous les step_size epochs"""
    return base_lr * (gamma ** (epoch // step_size))

def lr_cosine(epoch, base_lr=0.1, T_max=100):
    """Cosine annealing"""
    return base_lr * 0.5 * (1 + np.cos(np.pi * epoch / T_max))

def lr_warmup_cosine(epoch, base_lr=0.1, warmup=10, T_max=100):
    """Warmup lineaire puis cosine (utilise dans les transformers)"""
    if epoch < warmup:
        return base_lr * epoch / warmup
    return base_lr * 0.5 * (1 + np.cos(np.pi * (epoch - warmup) / (T_max - warmup)))

def lr_one_cycle(epoch, base_lr=0.01, max_lr=0.1, total_steps=100):
    """1Cycle policy: lr monte puis descend"""
    mid = total_steps // 2
    if epoch < mid:
        return base_lr + (max_lr - base_lr) * epoch / mid
    return max_lr - (max_lr - base_lr) * (epoch - mid) / mid

print("=== Learning Rate Schedulers ===")
print(f"  {'Epoch':>6}  {'Step':>8}  {'Cosine':>8}  {'Warmup+Cos':>12}  {'OneCycle':>10}")
for e in [0, 5, 10, 20, 30, 50, 70, 90, 99]:
    print(f"  {e:6d}  {lr_step(e):.6f}  {lr_cosine(e):.6f}  {lr_warmup_cosine(e):.6f}  {lr_one_cycle(e):.6f}")

print("\n  => Step decay: brutal, facile a implementer")
print("  => Cosine: smooth, souvent meilleur")
print("  => Warmup+Cosine: standard pour les transformers (evite instabilite initiale)")
print("  => OneCycle: tres efficace (super-convergence)")


---
## 5. Régularisation <a name='regularization'></a>

### Techniques cles

**Dropout**
- Masque aleatoirement p% des neurones pendant l'entrainement
- Au test: tous les neurones actifs, poids *multiplies* par (1-p) (ou scaling a l'entrainement = inverted dropout)
- Reduit le co-adaptation des neurones => regularisation implicite

**Batch Normalization**
- Normalise les activations d'un mini-batch : `z_hat = (z - mu) / sigma`
- Puis scale et shift apprenables : `y = gamma * z_hat + beta`
- Avantages : entraine plus vite, permet lr plus eleve, regularise legerement
- **Attention** : comportement different train vs inference (running stats)

**Layer Normalization**
- Normalise sur les features (pas le batch) => stable avec batch_size=1
- Utilise dans les **Transformers** (pas BatchNorm)

**Early Stopping**
- Arreter l'entrainement quand la validation loss cesse de diminuer
- Moyen simple et efficace d'eviter l'overfitting

**L1/L2 Regularisation**
- L2 (Weight Decay) : `loss += lambda * sum(w^2)` => shrink vers 0
- L1 : `loss += lambda * sum(|w|)` => sparsity


In [ ]:
import numpy as np

# ============================================================
# DROPOUT -- from scratch (inverted dropout)
# ============================================================
class Dropout:
    """Inverted dropout : scale pendant l entrainement, rien au test"""
    def __init__(self, p=0.5):
        self.p = p          # probabilite de desactivation
        self.mask = None
        self.training = True

    def forward(self, x):
        if not self.training:
            return x  # pas de dropout au test
        # Masque binaire + scaling (inverted dropout)
        self.mask = (np.random.rand(*x.shape) > self.p) / (1.0 - self.p)
        return x * self.mask

    def backward(self, dout):
        return dout * self.mask  # meme masque qu au forward

np.random.seed(42)
x = np.random.randn(4, 8)
dropout = Dropout(p=0.5)
dropout.training = True

print("=== Dropout (training) ===")
out_train = dropout.forward(x)
print(f"  Fraction zeros : {(out_train == 0).mean():.2f}  (attendu ~0.50)")
print(f"  Mean input     : {x.mean():.4f}")
print(f"  Mean output    : {out_train.mean():.4f}  (doit rester proche)")

dropout.training = False
out_test = dropout.forward(x)
print(f"\n=== Dropout (inference) ===")
print(f"  Mean test output : {out_test.mean():.4f}  (identique a l entree)")

# ============================================================
# BATCH NORMALIZATION -- from scratch
# ============================================================
class BatchNorm1d:
    def __init__(self, n_features, eps=1e-5, momentum=0.1):
        self.gamma = np.ones(n_features)
        self.beta = np.zeros(n_features)
        self.eps = eps
        self.momentum = momentum
        # Running stats pour inference
        self.running_mean = np.zeros(n_features)
        self.running_var = np.ones(n_features)
        self.training = True
        self.cache = {}

    def forward(self, x):
        if self.training:
            mu  = x.mean(axis=0)
            var = x.var(axis=0)
            x_norm = (x - mu) / np.sqrt(var + self.eps)
            # MAJ running stats
            self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*mu
            self.running_var  = (1-self.momentum)*self.running_var  + self.momentum*var
            self.cache = {"x": x, "mu": mu, "var": var, "x_norm": x_norm}
        else:
            # Utilise les stats enregistrees
            x_norm = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)

        return self.gamma * x_norm + self.beta

print("\n=== Batch Normalization ===")
np.random.seed(0)
batch = np.random.randn(32, 10) * 5 + 3  # donnees non centrees
bn = BatchNorm1d(10)

out_bn = bn.forward(batch)
print(f"  Input  - mean: {batch.mean(axis=0)[:3].round(2)}, std: {batch.std(axis=0)[:3].round(2)}")
print(f"  Output - mean: {out_bn.mean(axis=0)[:3].round(4)}, std: {out_bn.std(axis=0)[:3].round(4)}")
print(f"  (gamma=1, beta=0 => sortie normalisee)")

print("\n=== BatchNorm vs LayerNorm ===")
print("  BatchNorm: normalise sur le batch (axe=0) -> instable si batch_size petit")
print("  LayerNorm: normalise sur les features (axe=-1) -> stable meme batch_size=1")
print("  => BatchNorm: CNN, MLP")
print("  => LayerNorm: Transformers, RNN")


In [ ]:
import numpy as np

# ============================================================
# LAYER NORMALIZATION -- utilise dans les Transformers
# ============================================================
class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps

    def forward(self, x):
        # Normalise sur la derniere dimension (features)
        mu  = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        x_norm = (x - mu) / np.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

np.random.seed(0)
x = np.random.randn(4, 8) * 3 + 2   # batch=4, seq=8
ln = LayerNorm(8)
out = ln.forward(x)
print("=== Layer Normalization ===")
print(f"  Input row 0  - mean: {x[0].mean():.3f}, std: {x[0].std():.3f}")
print(f"  Output row 0 - mean: {out[0].mean():.6f}, std: {out[0].std():.6f}")

# ============================================================
# EARLY STOPPING -- implementation
# ============================================================
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4, restore_best=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best = restore_best
        self.best_loss = float("inf")
        self.best_weights = None
        self.counter = 0
        self.stopped_epoch = 0

    def __call__(self, val_loss, weights=None):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            if self.restore_best:
                self.best_weights = weights
            return False   # continue
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True  # stop!
            return False

print("\n=== Early Stopping ===")
es = EarlyStopping(patience=3)
# Simulation d un entrainement
val_losses = [0.9, 0.7, 0.5, 0.4, 0.42, 0.44, 0.45, 0.6]
for epoch, loss in enumerate(val_losses):
    stop = es(loss)
    print(f"  Epoch {epoch+1}: val_loss={loss:.2f}, counter={es.counter}, stop={stop}")
    if stop:
        print(f"  => Early stopping a l epoch {epoch+1}! Meilleure loss: {es.best_loss:.2f}")
        break

print("\n=== Resume des techniques de regularisation ===")
print("  L2 (Weight Decay) : reduit la magnitude des poids, solution lisse")
print("  L1                : sparsity, feature selection implicite")
print("  Dropout           : reduit co-adaptation, agit comme ensemble de modeles")
print("  BatchNorm         : acceleration + legere regularisation")
print("  Early Stopping    : simple, efficace, pas de surcout de parametres")
print("  Data Augmentation : la meilleure regularisation si on peut generer de la data")


---
## 6. CNN — Convolutional Neural Networks <a name='cnn'></a>

### Concepts fondamentaux

**Convolution 2D**
- Filtre de taille `(k, k)` glisse sur l'image
- Output size : `H' = (H - k + 2*padding) / stride + 1`
- Nombre de params : `C_out * (C_in * k * k + 1)`

**Pooling**
- **MaxPooling** : prend le max dans chaque fenetre (localisation robuste)
- **AvgPooling** : moyenne (plus lisse, utilise dans GAP)
- **Global Average Pooling (GAP)** : remplace Flatten -> FC, reduit l overfitting

**Inductive biases des CNN**
- **Translation equivariance** : meme feature detectee peu importe la position
- **Local connectivity** : chaque neurone ne voit qu une region locale
- **Weight sharing** : meme filtre partage sur toute l image

### Architectures de reference
| Architecture | Innovation | Annee |
|---|---|---|
| LeNet-5 | Premiere CNN moderne | 1998 |
| AlexNet | ReLU, Dropout, GPU | 2012 |
| VGG | Petits filtres 3x3 empiles | 2014 |
| ResNet | Skip connections, 152 couches | 2015 |
| EfficientNet | Scaling compound | 2019 |


In [ ]:
import numpy as np

# ============================================================
# CONVOLUTION 2D -- from scratch
# ============================================================
def conv2d(x, kernel, stride=1, padding=0):
    """
    x      : (H, W) image
    kernel : (kH, kW) filtre
    """
    if padding > 0:
        x = np.pad(x, padding, mode="constant")

    H, W   = x.shape
    kH, kW = kernel.shape
    H_out  = (H - kH) // stride + 1
    W_out  = (W - kW) // stride + 1

    out = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            patch = x[i*stride:i*stride+kH, j*stride:j*stride+kW]
            out[i, j] = np.sum(patch * kernel)
    return out

# Test avec un filtre de detection de contours
img = np.array([
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 1, 2, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0],
], dtype=float)

# Sobel horizontal (detection de bords)
sobel_h = np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=float)
# Sobel vertical
sobel_v = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=float)
# Gaussian blur
gaussian = np.array([[1,2,1],[2,4,2],[1,2,1]], dtype=float) / 16

print("=== Convolution 2D from scratch ===")
print(f"Input (5x5):\n{img}")
print(f"\nSobel H (bords horizontaux):\n{conv2d(img, sobel_h, padding=1)}")
print(f"\nSobel V (bords verticaux):\n{conv2d(img, sobel_v, padding=1)}")
print(f"\nGaussian blur:\n{conv2d(img, gaussian, padding=1).round(2)}")


In [ ]:
import numpy as np

# ============================================================
# OUTPUT SIZE -- calcul des dimensions
# ============================================================
def conv_output_size(H, k, padding=0, stride=1, dilation=1):
    k_eff = k + (k-1)*(dilation-1)  # taille effective avec dilation
    return (H + 2*padding - k_eff) // stride + 1

print("=== Taille de sortie d une convolution ===")
print(f"  H' = (H + 2*padding - kernel) // stride + 1")
print()
configs = [
    (224, 3, 0, 1, "Standard 3x3, no pad"),
    (224, 3, 1, 1, "Standard 3x3, same pad"),
    (224, 5, 2, 1, "5x5, same pad"),
    (224, 3, 0, 2, "3x3, stride 2 (downsampling)"),
    (56,  3, 1, 1, "3x3, same (feature map 56)"),
]
print(f"  {'Description':30s}  {'In':>4}  {'k':>3}  {'pad':>4}  {'stride':>6}  {'Out':>4}")
for H, k, p, s, desc in configs:
    H_out = conv_output_size(H, k, p, s)
    print(f"  {desc:30s}  {H:>4}  {k:>3}  {p:>4}  {s:>6}  {H_out:>4}")

# ============================================================
# CALCUL DES PARAMETRES
# ============================================================
print("\n=== Nombre de parametres par couche ===")
configs_params = [
    ("Conv2D(3, 64, 3x3)",      3,   64,  3),
    ("Conv2D(64, 128, 3x3)",   64,  128,  3),
    ("Conv2D(128, 256, 3x3)", 128,  256,  3),
    ("Conv2D(256, 512, 3x3)", 256,  512,  3),
]
for name, c_in, c_out, k in configs_params:
    params_w = c_out * c_in * k * k
    params_b = c_out
    total = params_w + params_b
    print(f"  {name:25s}: W={params_w:>8,}  b={params_b:>5}  total={total:>8,}")

# ============================================================
# SKIP CONNECTIONS (ResNet)
# ============================================================
print("\n=== ResNet Skip Connection (ResBlock) ===")
class ResBlock:
    def __init__(self, channels):
        # 2 couches conv 3x3
        self.W1 = np.random.randn(channels, channels, 3, 3) * 0.01
        self.W2 = np.random.randn(channels, channels, 3, 3) * 0.01

    def forward(self, x):
        # Simule: out = F(x) + x  (residual connection)
        F_x = x * 0.1  # simulation d une conv
        out = F_x + x  # skip connection!
        return np.maximum(0, out)  # ReLU apres addition

x = np.random.randn(4, 64, 28, 28)  # batch=4, C=64, H=W=28
block = ResBlock(64)
out = block.forward(x)
print(f"  Input:  {x.shape}")
print(f"  Output: {out.shape}  (meme taille grace au skip!)")
print("  => Gradient peut 'court-circuiter' le bloc => resout vanishing gradient")
print("  => ResNet permet 100+ couches avec gradient stable")


---
## 7. RNN, LSTM, GRU <a name='rnn'></a>

### RNN (Recurrent Neural Network)
`h_t = tanh(W_h * h_{t-1} + W_x * x_t + b)`

- Partage des poids dans le temps
- **Probleme** : vanishing/exploding gradient sur longues sequences (BPTT)
- Utilisation : sequences courtes

### LSTM (Long Short-Term Memory)
Introduit une **cell state** c_t (conveyor belt) controllee par 3 gates :
```
f_t = sigmoid(W_f * [h_{t-1}, x_t] + b_f)   # forget gate
i_t = sigmoid(W_i * [h_{t-1}, x_t] + b_i)   # input gate
o_t = sigmoid(W_o * [h_{t-1}, x_t] + b_o)   # output gate
g_t = tanh(W_g * [h_{t-1}, x_t] + b_g)      # candidate cell

c_t = f_t * c_{t-1} + i_t * g_t              # cell state update
h_t = o_t * tanh(c_t)                         # hidden state
```

### GRU (Gated Recurrent Unit)
- Simplifie LSTM : 2 gates au lieu de 3, pas de cell state separee
- Souvent aussi bon que LSTM, moins de parametres
```
z_t = sigmoid(W_z * [h_{t-1}, x_t])   # update gate
r_t = sigmoid(W_r * [h_{t-1}, x_t])   # reset gate
h_t = (1-z_t)*h_{t-1} + z_t*tanh(W*[r_t*h_{t-1}, x_t])
```


In [ ]:
import numpy as np

# ============================================================
# RNN CELL -- from scratch
# ============================================================
class RNNCell:
    def __init__(self, input_size, hidden_size):
        scale = np.sqrt(2.0 / (input_size + hidden_size))
        self.Wh = np.random.randn(hidden_size, hidden_size) * scale
        self.Wx = np.random.randn(hidden_size, input_size)  * scale
        self.b  = np.zeros(hidden_size)

    def forward(self, x, h_prev):
        # h_t = tanh(Wh * h_{t-1} + Wx * x_t + b)
        return np.tanh(self.Wh @ h_prev + self.Wx @ x + self.b)

class LSTMCell:
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        d = input_size + hidden_size
        scale = np.sqrt(1.0 / hidden_size)
        # 4 gates concatenes (f, i, o, g) pour efficacite
        self.W = np.random.randn(4 * hidden_size, d) * scale
        self.b = np.zeros(4 * hidden_size)

    def forward(self, x, h_prev, c_prev):
        xh = np.concatenate([x, h_prev])
        gates = self.W @ xh + self.b

        H = self.hidden_size
        f = self._sigmoid(gates[0*H:1*H])      # forget gate
        i = self._sigmoid(gates[1*H:2*H])      # input gate
        o = self._sigmoid(gates[2*H:3*H])      # output gate
        g = np.tanh(gates[3*H:4*H])            # candidate cell

        c = f * c_prev + i * g                  # cell state
        h = o * np.tanh(c)                      # hidden state
        return h, c, {"f": f, "i": i, "o": o, "g": g}

    def _sigmoid(self, z): return 1/(1+np.exp(-np.clip(z,-500,500)))

class GRUCell:
    def __init__(self, input_size, hidden_size):
        self.H = hidden_size
        d = input_size + hidden_size
        scale = np.sqrt(1.0 / hidden_size)
        self.Wz = np.random.randn(hidden_size, d) * scale
        self.Wr = np.random.randn(hidden_size, d) * scale
        self.Wh = np.random.randn(hidden_size, d) * scale

    def _sigmoid(self, z): return 1/(1+np.exp(-np.clip(z,-500,500)))

    def forward(self, x, h_prev):
        xh = np.concatenate([x, h_prev])
        z = self._sigmoid(self.Wz @ xh)        # update gate
        r = self._sigmoid(self.Wr @ xh)        # reset gate
        xrh = np.concatenate([x, r * h_prev])
        h_cand = np.tanh(self.Wh @ xrh)        # candidate
        h = (1 - z) * h_prev + z * h_cand      # output
        return h

# ============================================================
# SIMULATION D UNE SEQUENCE
# ============================================================
T, input_size, hidden_size = 10, 5, 16
x_seq = [np.random.randn(input_size) for _ in range(T)]

rnn  = RNNCell(input_size, hidden_size)
lstm = LSTMCell(input_size, hidden_size)
gru  = GRUCell(input_size, hidden_size)

h_rnn = np.zeros(hidden_size)
h_lstm, c_lstm = np.zeros(hidden_size), np.zeros(hidden_size)
h_gru = np.zeros(hidden_size)

print("=== Forward pass sur sequence T=10 ===")
for t, x in enumerate(x_seq):
    h_rnn               = rnn.forward(x, h_rnn)
    h_lstm, c_lstm, g   = lstm.forward(x, h_lstm, c_lstm)
    h_gru               = gru.forward(x, h_gru)

print(f"  RNN  h_{T}: norm={np.linalg.norm(h_rnn):.4f}")
print(f"  LSTM h_{T}: norm={np.linalg.norm(h_lstm):.4f}, c norm={np.linalg.norm(c_lstm):.4f}")
print(f"  GRU  h_{T}: norm={np.linalg.norm(h_gru):.4f}")
print(f"  LSTM gates (dernier pas): f={g['f'].mean():.3f}, i={g['i'].mean():.3f}, o={g['o'].mean():.3f}")
print(f"  => forget gate proche de 1 => cellule garde l info")

print("\n=== Nb de parametres ===")
n_rnn  = hidden_size*(hidden_size+input_size) + hidden_size
n_lstm = 4*(hidden_size*(hidden_size+input_size) + hidden_size)
n_gru  = 3*(hidden_size*(hidden_size+input_size) + hidden_size)
print(f"  RNN  : {n_rnn:,}")
print(f"  GRU  : {n_gru:,}")
print(f"  LSTM : {n_lstm:,}")
print(f"  => GRU ~ 75% des params de LSTM pour des perfs similaires")


In [ ]:
import numpy as np

# ============================================================
# GRADIENT CLIPPING -- solution aux exploding gradients
# ============================================================
def clip_gradients_by_norm(grads, max_norm=1.0):
    """Clip les gradients si leur norme globale depasse max_norm"""
    total_norm = np.sqrt(sum(np.sum(g**2) for g in grads))
    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1.0:
        grads = [g * clip_coef for g in grads]
    return grads, total_norm

print("=== Gradient Clipping ===")
# Simulation de gradients explosifs (typique dans les RNNs)
gradients_normal   = [np.random.randn(10, 10) * 0.1 for _ in range(4)]
gradients_exploding = [np.random.randn(10, 10) * 100.0 for _ in range(4)]

g_clipped_n, norm_n = clip_gradients_by_norm(gradients_normal, max_norm=1.0)
g_clipped_e, norm_e = clip_gradients_by_norm(gradients_exploding, max_norm=1.0)

print(f"  Gradients normaux   - norm avant: {norm_n:.4f}, apres: {np.sqrt(sum(np.sum(g**2) for g in g_clipped_n)):.4f}")
print(f"  Gradients explosifs - norm avant: {norm_e:.4f}, apres: {np.sqrt(sum(np.sum(g**2) for g in g_clipped_e)):.4f}")
print("  => Avec clipping: la norme reste <= max_norm")

print("\n=== RNN vs LSTM vs GRU vs Transformer ===")
comparison = [
    ("RNN",         "Simple",           "Vanishing grad, memoire courte",      "Donnees simples courtes"),
    ("LSTM",        "Memoire longue terme", "Lourd (4x params RNN), lent",     "Series temp, NLP classique"),
    ("GRU",         "Rapide, 75% LSTM",  "Memoire un peu moins longue",        "Sequences, resource limitee"),
    ("Transformer", "Attention globale", "O(n^2) attention, beaucoup de data", "NLP moderne, vision (ViT)"),
]
print(f"  {'Modele':12s}  {'Avantage':25s}  {'Inconvenient':30s}  {'Cas d usage':25s}")
for model, adv, dis, use in comparison:
    print(f"  {model:12s}  {adv:25s}  {dis:30s}  {use:25s}")


---
## 8. Questions d’entretien <a name='questions'></a>

**Q : Expliquer la backpropagation en termes simples.**
> On calcule le gradient de la loss par rapport a chaque parametre en appliquant la regle de la chaine de la sortie vers l entree. On stocke les activations au forward pass, puis on les reutilise au backward pour calculer les gradients efficacement.

**Q : Pourquoi BatchNorm accelere l entrainement ?**
> En normalisant les activations, on evite le Internal Covariate Shift : les distributions des entrees de chaque couche ne changent plus brutalement a chaque iteration. Cela permet d utiliser un learning rate plus eleve et reduit la dependance a l initialisation.

**Q : Quelle est la difference entre BatchNorm et LayerNorm ?**
> BatchNorm normalise sur le batch (axe=0), instable si batch_size petit. LayerNorm normalise sur les features (axe=-1), stable meme avec batch_size=1. => BatchNorm pour CNN/MLP, LayerNorm pour Transformers.

**Q : Expliquer le probleme du vanishing gradient et les solutions.**
> Dans un reseau profond avec sigmoid/tanh, le gradient du produit de N derivees (< 0.25) -> 0 exponentiellement. Solutions : (1) ReLU/GELU, (2) BatchNorm, (3) skip connections (ResNet), (4) LSTM pour les RNN, (5) gradient clipping.

**Q : Pourquoi ReLU peut creer des neurones morts ?**
> Si z < 0, ReLU sort 0 et son gradient est 0. Si un neurone reçoit toujours des entrees negatives, il n apprend plus jamais. Solutions : Leaky ReLU (alpha=0.01), ELU, ou choisir un bon learning rate.

**Q : Quelle est l intuition derriere les skip connections de ResNet ?**
> `F(x) + x` : au lieu d apprendre la transformation complete, le reseau apprend la residuelle (difference). Si F(x) = 0, on a une identite => le reseau peut facilement apprendre des transformations proches de l identite. Resout le vanishing gradient car le gradient peut passer directement par le chemin +x.

**Q : LSTM vs GRU : quand choisir lequel ?**
> GRU : moins de parametres, plus rapide, souvent aussi bon sur des datasets de taille moyenne. LSTM : sequences tres longues, quand la memoire a long terme est critique. Dans la pratique : commencer par GRU, passer a LSTM si les perfs sont insuffisantes.

**Q : Qu est-ce que l inverted dropout ?**
> On divise les neurones actifs par (1-p) pendant l entrainement pour que l esperance reste la meme qu au test. Ainsi, au test on n a pas besoin de modifier les poids. C est l implementation standard (PyTorch, TensorFlow).
